## RAG Pipelines - Data Ingestion to VectorDB Pipelines

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Abhimanyu  Singh\AppData\Local\Temp\ipykernel_31616\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\Abhimanyu  Singh\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# now we create a finction to read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    # processing all pdf files in the directory
    all_docs = []
    pdf_dir = Path(pdf_directory)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nprocessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source info to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            
            all_docs.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error {e}")
    
    print(f"total documents loaded: {len(all_docs)}")
    return all_docs

# process all the pdfs in the data directory
all_pdf_docs = process_all_pdfs("../data")

found 2 PDF files to process

processing PE_Class 1.pdf
loaded 87 pages

processing PE_Class 2.pdf
loaded 33 pages
total documents loaded: 120


### Chunking

In [3]:
# now we will perform chunking
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    # split docs into smaller chunks for better rag performance
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if(split_docs):
        print(f"\nexample chunk:")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"metadata: {split_docs[0].metadata}")
    
    return split_docs

In [4]:
chunks = split_documents(all_pdf_docs)
chunks

split 120 documents into 122 chunks

example chunk:
content: Power Electronics
6th Sem, ECE, NIT Silchar...
metadata: {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2026-02-02T17:00:35+05:30', 'title': 'Power Electronics', 'author': 'ECE', 'moddate': '2026-02-02T17:00:35+05:30', 'source': '..\\data\\pdfs\\PE_Class 1.pdf', 'total_pages': 87, 'page': 0, 'page_label': '1', 'source_file': 'PE_Class 1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2026-02-02T17:00:35+05:30', 'title': 'Power Electronics', 'author': 'ECE', 'moddate': '2026-02-02T17:00:35+05:30', 'source': '..\\data\\pdfs\\PE_Class 1.pdf', 'total_pages': 87, 'page': 0, 'page_label': '1', 'source_file': 'PE_Class 1.pdf', 'file_type': 'pdf'}, page_content='Power Electronics\n6th Sem, ECE, NIT Silchar'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2026-02-02T17:00:35+05:30', 'title': 'Power Electronics', 'author': 'ECE', 'moddate': '2026-02-02T17:00:35+05:30', 'source': '..\\data\\pdfs\\PE_Class 1.pdf', 'total_pages': 87, 'page': 1, 'page_label': '2', 'source_file': 'PE_Class 1.pdf', 'file_type': 'pdf'}, page_content='Unit 1 Introduction: need for power conversion with efficient, high frequency, light weight \nconverters; Power electronic converters classifications and sc

### Embeddings and VectorDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer #embedding model inside this
import chromadb
from chromadb.config import Settings
import uuid #id of each entry in the vector db
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingsManager:
    # handles document embedding generation using sentence transformers

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        """intitialize the embedding manager
        
        model_name -> huggingface model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()    #used for loading the abive model having tha name model_name

    def _load_model(self):
        # loads the sentence tansformer model

        try:
            print(f"LOADING THE MODEL : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"MODEL LOADED SUCCESSFULLY, EMBEDDING DIMENSION : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"ERROR LOADING THE MODEL : {self.model_name} : {e}")
            raise

    def generateEmbeddings(self, texts:List[str]) ->np.ndarray:
        """generate embeddings for a list of texts"""

        """texts - list of texts to embed and returnns a numpy array"""

        if not self.model:
            raise ValueError("MODEL NOT LOADED!")

        print(f"GENERATING EMBEDDINGS FOR {len(texts)} TEXTS...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"GENERATED EMBEDDINGS WITH SHAPE : {embeddings.shape}")
        return embeddings


# initialize the embedding manager
embedding_manager = EmbeddingsManager()
embedding_manager

LOADING THE MODEL : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2546.90it/s]


MODEL LOADED SUCCESSFULLY, EMBEDDING DIMENSION : 384


In [7]:
# vector db
class VectorStore:
    """Manages embddings in a ChromaDB vector store"""
    def __init__(self, collection_name : str = "pdf_documents", persist_directory : str = "../data/vector_store"):
        """intitialize the vector store class"""

        """collection_name -> name of the chroma db vector store
            persist_directory -> diretory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        pass

    def _initialize_store(self):
        """Initialie ChromaDB client and collection"""

        try:
            # initialize chroma db client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name= self.collection_name,
                metadata={"descripton" : "PDF Document embeddings for RAG"}
            )
            print(f"Vector Store Initialized, Collection : {self.collection_name}")
            print(f"Existing Documents in Collection : {self.collection.count()}")

        except Exception as e:
            print(f"Error Initializing Vector Store : {e}")
            raise

    def addDocuments(self, documents : List[Any], embeddings : np.ndarray):
        """add documents and their embeddings to the vector store
            documents -> list of langchain documents
            embeddings -> corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be equal!")

        print(f"Adding {len(embeddings)} embeddings in the vector store...")

        # prepare data for chroma db
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())

        # add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection : {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to the vector store : {e}")
            raise


# initializing the vector store 
vectorstore = VectorStore()
vectorstore

Vector Store Initialized, Collection : pdf_documents
Existing Documents in Collection : 0


In [8]:
# now we convert the text present in the chunks into embeddings
texts = [doc.page_content for doc in chunks]

# generate embeddings
embeddings = embedding_manager.generateEmbeddings(texts)

# store the embeddings in the vector store
vectorstore.addDocuments(chunks, embeddings)

GENERATING EMBEDDINGS FOR 122 TEXTS...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.98it/s]


GENERATED EMBEDDINGS WITH SHAPE : (122, 384)
Adding 122 embeddings in the vector store...
Successfully added 122 documents to vector store
Total documents in collection : 122


In [9]:
# upto this point we have created the data ingestion pipeline
# now we can start with the data retrieval pipeline

## Retriever Pipeline from Vector Store

In [11]:
class RAGRetriever:

    """handles query based retrieval from the vector store"""
    def __init__(self, vector_store : VectorStore, embedding_manager : EmbeddingsManager):
        """initialize the retriever"""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query : str, top_k : int = 5, score_threshold : float = 0.0) -> List[Dict[str, Any]]:
        """retrieve relevant documents for a query"""
        """query : the search query
           top_k : no. of top results to return
           score_threshold : minimum similarity score threshold"""

        """returns : list of dictionaries of retrived information and metadata"""

        print(f"Retrieving documents for query : {query}")
        print(f"Top k : {top_k}, Score Threshold : {score_threshold}")

        # generate query embedding
        query_embedding = self.embedding_manager.generateEmbeddings([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids']

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score, chroma db uses cosine similarity
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents founnd!")

            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving the documents : {e}")
            raise

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [14]:
rag_retriever.retrieve("what is reverse recovery time")

Retrieving documents for query : what is reverse recovery time
Top k : 5, Score Threshold : 0.0
GENERATING EMBEDDINGS FOR 1 TEXTS...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.37it/s]

GENERATED EMBEDDINGS WITH SHAPE : (1, 384)
Retrieved 1 documents (after filtering)


[{'id': ['doc_da755ef2_25',
   'doc_da5bdcc4_26',
   'doc_3ec5f6bf_96',
   'doc_fce25733_24',
   'doc_6d5004c0_28'],
  'content': 'Reverse Recovery Power Diode IV characteristics\n• Reverse recovery time: The reverse recovery time (𝑡𝑟𝑟) is defined as the time between the instant the\nforward diode current becomes zero and the instant reverse recovery current decays to 25% of its reverse\npeak value𝐼𝑅𝑀\n• The reverse recovery time(𝑡𝑟𝑟) is composed of two segments of time𝑡𝑎 and𝑡𝑏\n𝑡𝑟𝑟=𝑡𝑎+𝑡𝑏\n• 𝑡𝑎 is the time between zero crossing of the  forward current and peak reverse current 𝐼𝑅𝑀. During  𝑡𝑎, stored \ncharge from depletion region is removed\n• 𝑡𝑏 is measured from the instant of peak reverse current 𝐼𝑅𝑀 to the instant when 25% of the peak reverse current \n𝐼𝑅𝑀 is reached. During  𝑡𝑏, stored charge from semiconductor region is removed\n• The ratio \n𝑡𝑏\n𝑡𝑎\nis called softness factor or S-factor\n• A diode with S-factor equal to 1 is called soft recovery diode and a diode with S-factor le

In [15]:
# so as seen above now we can retrieve the context from the vector store as well based on the user query, and it is
# this context that we provide a LLM along with a prompt

### Integrating VectorDB context pipeline with LLM Output

In [19]:
# Creating a simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

# create a simple RAG function - retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc['content'] for doc in results) if results else ""

    if not context:
        return "No relevant context found to answer the question"

    prompt = f"""Use the following context to answer the question consisely
        Context : {context}
        Question : {query}
        Answer :"""
    response = llm.invoke([prompt.format(context=context,query=query)])

    return response.content

In [20]:
answer = rag_simple("what is reverse recovery time?", rag_retriever, llm)
print(answer)

Retrieving documents for query : what is reverse recovery time?
Top k : 3, Score Threshold : 0.0
GENERATING EMBEDDINGS FOR 1 TEXTS...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.55it/s]

GENERATED EMBEDDINGS WITH SHAPE : (1, 384)
Retrieved 1 documents (after filtering)


The reverse recovery time (𝑡𝑟𝑟) is defined as the time between the instant the forward diode current becomes zero and the instant reverse recovery current decays to 25% of its reverse peak value 𝐼𝑅𝑀.


In [21]:
answer = rag_simple("what is the difference between turn on time and delay time?", rag_retriever, llm)
print(answer)

Retrieving documents for query : what is the difference between turn on time and delay time?
Top k : 3, Score Threshold : 0.0
GENERATING EMBEDDINGS FOR 1 TEXTS...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.45it/s]

GENERATED EMBEDDINGS WITH SHAPE : (1, 384)
Retrieved 1 documents (after filtering)


The difference between turn-on time and delay time is that turn-on time is the total time required for a thyristor to change from the forward blocking state to the final on-state, while delay time (𝑡𝑑) is a specific interval within the turn-on time, representing the time between the application of the gate voltage and the start of the current flow.


In [24]:
answer = rag_simple("what is rise time and how does it work?", rag_retriever, llm)
print(answer)

Retrieving documents for query : what is rise time and how does it work?
Top k : 3, Score Threshold : 0.0
GENERATING EMBEDDINGS FOR 1 TEXTS...


Batches: 100%|██████████| 1/1 [00:00<00:00, 73.87it/s]

GENERATED EMBEDDINGS WITH SHAPE : (1, 384)
Retrieved 1 documents (after filtering)


Rise Time (𝑡𝑟) is the time taken by the anode current to rise from 0.1𝐼𝑎 to 0.9𝐼𝑎, where 𝐼𝑎 is the final value of anode current. It is inversely proportional to the magnitude of gate current and its build-up rate. The nature of the anode circuit, such as series RL or RC circuits, also affects the rise time. A higher and steeper current pulse applied to the gate can reduce the rise time.


In [25]:
# so above is a simple rag pipeline which returns simple answers based on the prompt and the context we provide to the llm
# we can also create enhanced veersion for this rag_simple function which will return not only the answer but information
# like - sources, confidence score, etc